# 01a · Image Basics `[local]`

**Goal:** understand exactly what an image is in code, and do the everyday operations with OpenCV: color spaces, histograms, contrast, drawing, and geometric transforms.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Images as grids of numbers |
| 📐 Math | Indexing, data types, color conversion, transforms as matrices |
| 🔨 From scratch | Build an image, grayscale, flips, brightness/contrast, histogram, resize, all with NumPy |
| 🚀 OpenCV | Color spaces, color-based masking, equalization/CLAHE, drawing, rotate/affine/perspective |
| 📊 Evaluate | Check your code against OpenCV; measure resize quality with PSNR |
| 🧪 Experiment | Predict-then-run challenges |
| 📝 Recap | Cheat sheet and interview questions |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data as skdata
from cvlib.data import opencv_sample
from cvlib.viz import show

---
# 🧠 1 · Intuition: an image is a spreadsheet of brightness

Zoom into any photo far enough and you see little squares: **pixels**. Each pixel is just a number.

- A **grayscale** image is one spreadsheet: each cell holds a brightness from **0 (black)** to **255 (white)**.
- A **color** image is **three spreadsheets stacked**: one for red, one for green, one for blue. Mixing the three brightnesses produces every color.

Everything in computer vision, from blurring to deep learning, is math on these grids. Let's look at a real one up close.

In [ ]:
gray = skdata.camera()                       # a classic 512×512 grayscale photo
patch = gray[200:212, 210:222]               # a tiny 12×12 window

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(gray, cmap="gray"); axes[0].add_patch(plt.Rectangle((210, 200), 12, 12, fill=False, color="red", lw=2))
axes[0].set_title("Full image (red box = zoomed area)"); axes[0].axis("off")
axes[1].imshow(patch, cmap="gray", vmin=0, vmax=255)
for (r, c), v in np.ndenumerate(patch):
    axes[1].text(c, r, v, ha="center", va="center", fontsize=7, color="red" if v > 128 else "yellow")
axes[1].set_title("Zoomed in: it's just numbers"); axes[1].axis("off")
plt.tight_layout(); plt.show()

---
# 📐 2 · The math you need

### Shape and indexing
OpenCV images are NumPy arrays with shape **`(height, width, channels)`**:

```
img[y, x]        → pixel at row y, column x   (row first! y before x)
img[y, x, c]     → one channel of that pixel
img[y1:y2, x1:x2] → a crop
```

⚠️ **The #1 beginner bug:** NumPy indexes `[row, col]` = `[y, x]`, but OpenCV *functions* take points and sizes as `(x, y)` and `(width, height)`. Two conventions, one image.

### Data types
Images are usually **`uint8`**: whole numbers 0–255, 1 byte each. Math on uint8 can **wrap around**:

$$200 + 100 = 300 \;\rightarrow\; 300 - 256 = 44 \quad\text{(NumPy uint8)}$$

OpenCV functions like `cv2.add` **saturate** instead (clip to 255). For any serious math, convert to `float32` first and convert back at the end.

### Color → grayscale
Our eyes are most sensitive to green and least to blue, so grayscale is a *weighted* average:

$$\text{gray} = 0.299\,R + 0.587\,G + 0.114\,B$$

### Geometric transforms are matrix multiplications
Moving, rotating, scaling and shearing a point $(x, y)$ is a small matrix multiplication:

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = \underbrace{\begin{bmatrix} a & b & t_x \\ c & d & t_y \end{bmatrix}}_{\text{2×3 affine matrix}} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

- **Affine** (2×3, 6 numbers): rotation, scale, shear, translation. Parallel lines stay parallel. Needs **3 point pairs**.
- **Perspective / homography** (3×3, 8 numbers): also handles "viewed at an angle". Parallel lines can meet. Needs **4 point pairs**.

---
# 🔨 3 · Build it from scratch (NumPy only)

### 3.1 · Make an image out of nothing

In [ ]:
canvas = np.zeros((200, 300, 3), dtype=np.uint8)   # height=200, width=300, 3 channels, all black
canvas[:, :100] = (255, 0, 0)        # left third
canvas[:, 100:200] = (255, 255, 255) # middle third
canvas[:, 200:] = (0, 0, 255)        # right third
show(canvas, "Which flag is this? (blue, white, red... or is it?)", size=4)

We set the left third to `(255, 0, 0)` expecting **red**, but `show()` displays it as **blue**. Why?

> **The BGR trap:** OpenCV stores channels in **B, G, R** order (a historical choice from the camera hardware of the time). Almost everything else (matplotlib, PIL, PyTorch models trained on ImageNet) expects **R, G, B**.
> `cvlib.viz.show` assumes BGR, like `cv2.imread` output, so `(255, 0, 0)` is pure blue.

Rule of thumb: **convert once, right after loading**, if your next tool expects RGB.

In [ ]:
img = cv2.imread(str(opencv_sample("fruits.jpg")))       # BGR
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img); axes[0].set_title("BGR array given to matplotlib (wrong)")
axes[1].imshow(rgb); axes[1].set_title("Converted to RGB (right)")
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

### 3.2 · Channels, grayscale and slicing tricks

In [ ]:
b, g, r = img[:, :, 0], img[:, :, 1], img[:, :, 2]
show([b, g, r], ["Blue channel", "Green channel", "Red channel"], cols=3, size=3.5)
print("Bright = lots of that color. The red/orange fruit is bright in the red channel and dark in blue.")

In [ ]:
def to_gray(img_bgr):
    """Weighted sum of channels. Note the BGR order of the weights."""
    weights = np.array([0.114, 0.587, 0.299])           # B, G, R
    gray = img_bgr.astype(np.float32) @ weights         # (H, W, 3) @ (3,) → (H, W)
    return np.clip(np.round(gray), 0, 255).astype(np.uint8)

my_gray = to_gray(img)
cv_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("max difference vs OpenCV:", np.abs(my_gray.astype(int) - cv_gray.astype(int)).max(), "(off by 1 from rounding is fine)")

h, w = img.shape[:2]
crop = img[h // 4: 3 * h // 4, w // 4: 3 * w // 4]       # center crop: plain slicing
flip_lr = img[:, ::-1]                                  # reverse the columns
flip_ud = img[::-1, :]                                  # reverse the rows
rot90 = np.rot90(img)                                   # transpose + flip
show([my_gray, crop, flip_lr, flip_ud, rot90], ["my grayscale", "center crop", "flip left-right", "flip up-down", "rotate 90°"], cols=5, size=2.6)

### 3.3 · Brightness, contrast, and the overflow trap

Brightness and contrast form a straight line: $\text{out} = \alpha \cdot \text{in} + \beta$
- $\beta$ (brightness) shifts everything up or down.
- $\alpha$ (contrast) stretches values away from 0 (>1 = more contrast).

In [ ]:
x = np.array([[200]], dtype=np.uint8)                   # a 1×1 "image"
y = np.array([[100]], dtype=np.uint8)
print("NumPy uint8 : 200 + 100 =", (x + y)[0, 0], " ← wrapped around!")
print("cv2.add     : 200 + 100 =", cv2.add(x, y)[0, 0], " ← saturated at 255")

def adjust(img, alpha=1.0, beta=0.0):
    """Contrast (alpha) and brightness (beta), done safely in float, then clipped."""
    out = img.astype(np.float32) * alpha + beta
    return np.clip(out, 0, 255).astype(np.uint8)

wrong = img + np.uint8(80)                               # the overflow bug
show([img, adjust(img, beta=80), wrong, adjust(img, alpha=1.6, beta=-60)],
     ["original", "brighter (safe)", "brighter (uint8 overflow bug!)", "more contrast"], cols=4, size=3)

### 3.4 · Histogram: how many pixels have each brightness?

A **histogram** counts how many pixels have value 0, 1, 2, ..., 255. It tells you at a glance whether an image is dark, bright, or low-contrast (everything bunched in a narrow range).

In [ ]:
moon = skdata.moon()                                     # a famously low-contrast image

my_hist = np.bincount(moon.ravel(), minlength=256)        # count each value 0..255
cv_hist = cv2.calcHist([moon], [0], None, [256], [0, 256]).ravel()
print("matches OpenCV:", np.array_equal(my_hist, cv_hist.astype(int)))
lo, hi = np.percentile(moon, [1, 99])
print(f"98% of pixels lie between {lo:.0f} and {hi:.0f}: about {hi - lo:.0f} of 256 possible levels")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].imshow(moon, cmap="gray", vmin=0, vmax=255); axes[0].axis("off"); axes[0].set_title("moon")
axes[1].bar(range(256), my_hist, width=1); axes[1].set_title("histogram (all bunched in the middle = low contrast)")
plt.tight_layout(); plt.show()

**Histogram equalization** spreads those bunched values over the full 0–255 range. The trick: map each value through the **cumulative distribution (CDF)**. If 30% of pixels are darker than value *v*, then *v* should become 30% of 255.

$$\text{new}(v) = 255 \cdot \frac{\text{CDF}(v) - \text{CDF}_{\min}}{N - \text{CDF}_{\min}}$$

In [ ]:
def equalize(gray):
    hist = np.bincount(gray.ravel(), minlength=256)
    cdf = hist.cumsum()
    cdf_min = cdf[cdf > 0][0]
    lut = np.round((cdf - cdf_min) / (cdf[-1] - cdf_min) * 255).clip(0, 255).astype(np.uint8)  # lookup table
    return lut[gray]                                       # apply the table to every pixel at once

my_eq = equalize(moon)
cv_eq = cv2.equalizeHist(moon)
print("max difference vs cv2.equalizeHist:", np.abs(my_eq.astype(int) - cv_eq.astype(int)).max())
show([moon, my_eq], ["original", "my equalization"], size=4)

### 3.5 · Resizing from scratch: nearest neighbor

To resize, every **output** pixel asks: *"where would I have come from in the input?"* and copies that pixel. (Going output → input avoids holes.)

In [ ]:
def resize_nearest(img, new_w, new_h):
    h, w = img.shape[:2]
    ys = (np.arange(new_h) * h / new_h).astype(int)       # source row for each output row
    xs = (np.arange(new_w) * w / new_w).astype(int)       # source col for each output col
    return img[ys[:, None], xs[None, :]]                  # fancy indexing builds the whole image at once

small = gray[180:240, 200:260]                             # 60×60 crop
big_mine = resize_nearest(small, 240, 240)
big_cv = cv2.resize(small, (240, 240), interpolation=cv2.INTER_NEAREST)   # note: (width, height)!
print("identical to OpenCV nearest:", np.array_equal(big_mine, big_cv))
show([small, big_mine], ["60×60 input", "4× nearest neighbor (blocky)"], size=4)

---
# 🚀 4 · The OpenCV way

### 4.1 · Color spaces: picking the right "coordinates" for color

RGB mixes *what color* with *how bright*, so a shadow changes all three numbers. Other color spaces separate them:

| Space | Channels | Great for |
|---|---|---|
| **HSV** | Hue (which color, 0–179 in OpenCV), Saturation (how vivid), Value (how bright) | Picking objects **by color**, robust to lighting |
| **LAB** | L = lightness, A = green↔red, B = blue↔yellow | Adjusting brightness without shifting colors; color distances |
| **Gray** | One brightness channel | Most classic CV algorithms |

⚠️ OpenCV hue runs **0–179** (not 0–359), so it fits in a uint8. Red sits at both ends: near 0 and near 179.

In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
H, S, V = cv2.split(hsv)
show([H, S, V], ["Hue (color identity)", "Saturation (vividness)", "Value (brightness)"], cols=3, size=3.5)

**Mini-project: find the oranges by color** (and *not* the yellow lemon next to them). Keep pixels with the right hue *and* enough saturation, so we ignore grayish background.

In [ ]:
lower, upper = np.array([5, 150, 80]), np.array([18, 255, 255])     # H, S, V ranges: orange hue, vivid, not too dark
mask = cv2.inRange(hsv, lower, upper)                                 # 255 where inside the range, else 0
only_fruit = cv2.bitwise_and(img, img, mask=mask)
show([img, mask, only_fruit], ["image", "mask: hue 5–18, saturation > 150", "masked result"], cols=3, size=3.5)

### 4.2 · Equalization vs CLAHE

Global equalization uses one mapping for the whole image, so it can blow out bright areas and amplify noise.
**CLAHE** (Contrast Limited Adaptive Histogram Equalization) equalizes **small tiles separately** and **limits** how much contrast it adds. It's the standard choice for medical images, documents and night photos.

For a color photo, apply it only to the **L channel of LAB**. Equalizing R, G, B separately would shift the colors.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
show([moon, cv2.equalizeHist(moon), clahe.apply(moon)], ["original", "global equalization", "CLAHE"], cols=3, size=3.5)

dark = adjust(img, alpha=0.5, beta=-20)                                # simulate an underexposed photo
lab = cv2.cvtColor(dark, cv2.COLOR_BGR2LAB)
lab[:, :, 0] = clahe.apply(lab[:, :, 0])                               # enhance lightness only
fixed = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
show([dark, fixed], ["underexposed", "CLAHE on the L channel of LAB"], size=4)

### 4.3 · Drawing: boxes, circles, lines and labels

You'll draw detections, keypoints and masks for the rest of the course. Drawing functions **modify the image in place**, so draw on a `.copy()`. Points are **(x, y)**.

In [ ]:
vis = img.copy()
ys, xs = np.nonzero(mask)                                               # pixel coordinates in our fruit mask
x1, x2 = np.percentile(xs, [1, 99]).astype(int)                         # percentiles ignore a few stray pixels
y1, y2 = np.percentile(ys, [1, 99]).astype(int)
cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)                  # green box (BGR!)
cv2.putText(vis, "orange pixels", (x1 + 5, y1 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
cv2.circle(vis, (int(xs.mean()), int(ys.mean())), 8, (0, 0, 255), -1)   # filled red dot at the center
cv2.line(vis, (0, 0), (int(xs.mean()), int(ys.mean())), (255, 0, 0), 2)
show(vis, "box around all masked pixels, center dot, a line from the corner")

### 4.4 · Resize, rotate, affine

**Interpolation**, i.e. how in-between pixels are computed when resizing:

| Flag | How | Use for |
|---|---|---|
| `INTER_NEAREST` | copy the closest pixel | label masks (must stay whole numbers) |
| `INTER_LINEAR` | blend 4 neighbors | general default, fast |
| `INTER_CUBIC` | blend 16 neighbors | enlarging photos (sharper) |
| `INTER_AREA` | average the covered area | **shrinking** (avoids jagged aliasing) |

In [ ]:
h, w = img.shape[:2]
M_rot = cv2.getRotationMatrix2D(center=(w / 2, h / 2), angle=30, scale=0.8)   # 2×3 affine matrix
rotated = cv2.warpAffine(img, M_rot, (w, h))                                    # output size is (width, height)
print("rotation matrix:\n", M_rot.round(3))

src = np.float32([[0, 0], [w - 1, 0], [0, h - 1]])                              # 3 points...
dst = np.float32([[0, 0], [w * 0.8, h * 0.1], [w * 0.2, h * 0.9]])              # ...and where they should go
sheared = cv2.warpAffine(img, cv2.getAffineTransform(src, dst), (w, h))
show([img, rotated, sheared], ["original", "rotate 30°, scale 0.8", "affine from 3 point pairs"], cols=3, size=3.5)

### 4.5 · Perspective transform: "view it from straight above"

A photo of a flat thing taken at an angle (a document, a sudoku, a whiteboard) can be **un-tilted** with a 3×3 perspective transform. Give 4 corners in the photo and where they should end up.

This is the core of the **document scanner** you'll build in 01b and use again in the OCR modules.

In [ ]:
sudoku = cv2.imread(str(opencv_sample("sudoku.png")))
corners = np.float32([[73, 85], [490, 70], [35, 512], [520, 520]])     # puzzle corners found by eye: TL, TR, BL, BR
side = 450
target = np.float32([[0, 0], [side - 1, 0], [0, side - 1], [side - 1, side - 1]])
H_persp = cv2.getPerspectiveTransform(corners, target)
flat = cv2.warpPerspective(sudoku, H_persp, (side, side))

vis = sudoku.copy()
cv2.polylines(vis, [corners[[0, 1, 3, 2]].astype(np.int32)], isClosed=True, color=(0, 0, 255), thickness=3)
show([vis, flat], ["4 corners picked by hand", "warped to a straight-on view"], size=4.5)
print("3×3 perspective matrix:\n", H_persp.round(3))

In 01b you'll find those 4 corners **automatically** with edges and contours.

---
# 📊 5 · Evaluate

For basic operations, "evaluation" means two things:
1. **Correctness:** does your code match a trusted implementation? (Done above: grayscale, histogram, equalization, nearest resize.)
2. **Quality:** which method gives the better result? Here, a number beats squinting at pictures.

**PSNR** (Peak Signal-to-Noise Ratio) measures how close an image is to a reference, in decibels. Higher is better, and +3 dB means roughly half the error.

$$\text{PSNR} = 10 \log_{10} \frac{255^2}{\text{MSE}}$$

**Test:** shrink an image 4× then enlarge it back, and compare to the original. Which interpolation loses the least?

In [ ]:
def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return float("inf") if mse == 0 else 10 * np.log10(255 ** 2 / mse)

ref = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
h, w = ref.shape
methods = {"NEAREST": cv2.INTER_NEAREST, "LINEAR": cv2.INTER_LINEAR, "CUBIC": cv2.INTER_CUBIC, "AREA": cv2.INTER_AREA, "LANCZOS4": cv2.INTER_LANCZOS4}

print(f"{'shrink with':<12}{'enlarge with':<14}{'PSNR (dB)':>10}")
results = {}
for down_name, down in [("NEAREST", cv2.INTER_NEAREST), ("AREA", cv2.INTER_AREA)]:
    small = cv2.resize(ref, (w // 4, h // 4), interpolation=down)
    for up_name in ["NEAREST", "LINEAR", "CUBIC", "LANCZOS4"]:
        back = cv2.resize(small, (w, h), interpolation=methods[up_name])
        results[(down_name, up_name)] = back
        print(f"{down_name:<12}{up_name:<14}{psnr(ref, back):>10.2f}")

In [ ]:
crop = (slice(150, 300), slice(150, 300))
show([ref[crop], results[("NEAREST", "NEAREST")][crop], results[("AREA", "CUBIC")][crop]],
     ["original", "nearest → nearest (worst)", "area → cubic"], cols=3, size=3.5)

**What the numbers say:**
- **Shrink with `INTER_AREA`**: nearest-neighbor shrinking simply throws pixels away, causing aliasing (jagged, shimmering detail).
- **Enlarge with `INTER_CUBIC` or `INTER_LANCZOS4`**: they keep edges smoother than nearest or linear.

This exact choice matters later: every neural network resizes its input, and bad resizing measurably lowers accuracy.

---
# 🧪 6 · Experiments: predict first, then run

Write your prediction in a comment **before** running each cell.

**Experiment 1: BGR mix-up in a "real" pipeline.** A model trained on RGB images is given BGR. What happens to a red stop sign as the model sees it?

In [ ]:
# Prediction: ...
stop_sign_bgr = np.zeros((100, 100, 3), np.uint8); stop_sign_bgr[:] = (0, 0, 220)   # red in BGR
seen_by_rgb_model = stop_sign_bgr                                                    # forgot to convert!
show([stop_sign_bgr, seen_by_rgb_model], ["what you see (BGR shown correctly)", "what an RGB model 'sees'"], bgr=False, size=2.5)
# The second panel is shown as if it were RGB. Did you predict blue?

**Experiment 2: HSV range.** Change `lower`/`upper` to select the **green** fruit instead. Hint: green hue is around 35–85 in OpenCV.

In [ ]:
# Prediction: which hue range isolates the green fruit/leaves?
lower_g, upper_g = np.array([35, 80, 40]), np.array([85, 255, 255])
mask_g = cv2.inRange(hsv, lower_g, upper_g)
show([mask_g, cv2.bitwise_and(img, img, mask=mask_g)], ["green mask", "green only"], size=3.5)

**Experiment 3: CLAHE clip limit.** What happens as `clipLimit` grows from 1 to 40? Predict before running.

In [ ]:
show([cv2.createCLAHE(clipLimit=c, tileGridSize=(8, 8)).apply(moon) for c in [1, 4, 40]],
     ["clipLimit=1", "clipLimit=4", "clipLimit=40"], cols=3, size=3.5)
# Higher clip limit = more local contrast, but also more amplified noise and texture.

**Experiment 4: rotation corners.** Rotating by 30° with `scale=1.0` cuts off the corners. Compute the output size needed to keep the whole image (hint: the rotated bounding box is `w·|cos| + h·|sin|` wide), and adjust the matrix's translation so the result is centered.

In [ ]:
def rotate_full(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w, new_h = int(w * cos + h * sin), int(w * sin + h * cos)
    M[0, 2] += new_w / 2 - w / 2                      # shift so the center lands in the new center
    M[1, 2] += new_h / 2 - h / 2
    return cv2.warpAffine(img, M, (new_w, new_h))

show([cv2.warpAffine(img, cv2.getRotationMatrix2D((w / 2, h / 2), 30, 1.0), (w, h)), rotate_full(img, 30)],
     ["corners cut off", "full image kept"], size=4)

---
# 📝 7 · Recap

**Cheat sheet**
- Image = NumPy array `(H, W, C)`, usually `uint8` 0–255. Index `img[y, x]`; OpenCV functions take `(x, y)` and `(width, height)`.
- `cv2.imread` returns **BGR**. Convert with `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` before matplotlib/PIL/PyTorch models.
- Do math in `float32`, then `np.clip(...).astype(np.uint8)`. Plain uint8 math wraps around.
- **HSV** to select by color (hue is 0–179), **LAB** to change brightness without changing colors.
- **CLAHE** beats global equalization for uneven lighting; apply it to L in LAB for color images.
- Resize: `INTER_AREA` to shrink, `INTER_CUBIC`/`LANCZOS4` to enlarge, `INTER_NEAREST` for label masks.
- Affine = 2×3 matrix from 3 point pairs. Perspective = 3×3 matrix from 4 point pairs.
- Drawing functions modify the image in place: draw on a `.copy()`.

**Interview-style questions**
1. Why does `cv2.imread` + `plt.imshow` show a blue face? How do you fix it?
2. What is `np.uint8(250) + np.uint8(10)`, and how do you avoid that bug?
3. Why is grayscale a weighted average and not `(R+G+B)/3`?
4. When would you pick HSV over RGB? Why does red need two hue ranges?
5. Which interpolation would you use to resize a segmentation mask, and why?
6. How many point pairs define an affine transform vs. a perspective transform, and why?
7. Why does CLAHE work better than global histogram equalization on a document photographed under a lamp?

**Next:** `01b_filters_and_shapes.ipynb`, where convolution, edges, thresholds and contours turn this sudoku warp into an automatic document scanner.